# **Assignment 1**

Last Edited: 9/6/2026 by Andrew

Notes: Setup SQL env and Finished Part A 1-5

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

# show a compact view of any result: first and last rows, plus its shape
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 120)

DB_PATH = Path("boilermaker_brews.db")

if not DB_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Place boilermaker_brews.db in the notebook's working folder."
        ) from exc

    print("Choose boilermaker_brews.db from the course files.")
    files.upload()

if not DB_PATH.exists():
    raise FileNotFoundError("boilermaker_brews.db was not uploaded.")

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")

def _execute_sql(connection, statement):
    statement = statement.strip()
    first_word = statement.split(None, 1)[0].upper()
    if first_word in {"SELECT", "WITH", "PRAGMA", "EXPLAIN"}:
        result = pd.read_sql_query(statement, connection)
        display(result)
        return

    connection.executescript(statement)
    connection.commit()
    print("Statement executed successfully.")

def _sql_magic(line, cell):
    _execute_sql(conn, cell)

def _expected_error_magic(line, cell):
    try:
        _execute_sql(conn, cell)
    except Exception as error:
        print(f"Expected error: {type(error).__name__}: {error}")
    else:
        raise AssertionError("This demonstration was expected to produce an SQL error.")

ip = get_ipython()
ip.register_magic_function(_sql_magic, "cell", "sql")
ip.register_magic_function(_expected_error_magic, "cell", "sql_expect_error")

table_count = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM sqlite_master WHERE type='table';", conn
).iloc[0, 0]
print(f"Connected to {DB_PATH.name}: {table_count} tables. The %%sql demo command is ready.")

Connected to boilermaker_brews.db: 6 tables. The %%sql demo command is ready.


**Initial Queries**

Epxloratory Data Analysis

In [ ]:
%%sql
SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;


,name
0,customers
1,employees
2,order_items
3,orders
4,products
5,stores


In [ ]:
%%sql
SELECT 'stores' AS table_name, COUNT(*) AS n_rows FROM stores
UNION ALL SELECT 'products',    COUNT(*) FROM products
UNION ALL SELECT 'customers',   COUNT(*) FROM customers
UNION ALL SELECT 'employees',   COUNT(*) FROM employees
UNION ALL SELECT 'orders',      COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items;


,table_name,n_rows
0,stores,6
1,products,26
2,customers,600
3,employees,36
4,orders,51927
5,order_items,78777


In [ ]:
%%sql
SELECT * FROM stores

,store_id,name,campus_area,opened_date,seats
0,1,Chauncey Hill,Chauncey,2019-08-12,38
1,2,PMU Ground Floor,Memorial Union,2017-01-09,64
2,3,Discovery Park,Discovery Park,2022-03-21,22
3,4,Levee Plaza,Levee,2020-09-01,30
4,5,State Street East,State Street,2018-05-14,26
5,6,Airport Rd Drive-Thru,South Campus,2023-10-02,0


In [ ]:
%%sql
SELECT * FROM stores s
JOIN employees e ON e.store_id = s.store_id
JOIN orders o ON o.employee_id = e.employee_id
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE s.store_id = 1

,store_id,name,campus_area,opened_date,seats,employee_id,name,store_id,hired_date,hourly_wage,...,channel,order_id,product_id,quantity,unit_price,product_id,name,category,price,is_seasonal
0,1,Chauncey Hill,Chauncey,2019-08-12,38,1,Tyler Park,1,2023-06-07,11.25,...,counter,1,10,1,5.25,10,Caramel Latte,espresso,5.25,0
1,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,2,16,1,2.75,16,Earl Grey Tea,tea,2.75,0
2,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,2,6,1,3.50,6,Americano,espresso,3.50,0
3,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,2,7,1,4.25,7,Cappuccino,espresso,4.25,0
4,1,Chauncey Hill,Chauncey,2019-08-12,38,4,Noah Shah,1,2026-03-03,14.77,...,counter,3,12,1,5.75,12,Pumpkin Spice Latte,espresso,5.75,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13430,1,Chauncey Hill,Chauncey,2019-08-12,38,4,Noah Shah,1,2026-03-03,14.77,...,app,51780,4,1,4.95,4,Nitro Cold Brew,brew,4.95,0
13431,1,Chauncey Hill,Chauncey,2019-08-12,38,4,Noah Shah,1,2026-03-03,14.77,...,app,51781,11,1,5.25,11,Mocha,espresso,5.25,0
13432,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,51782,9,1,5.00,9,Latte 16oz,espresso,5.00,0
13433,1,Chauncey Hill,Chauncey,2019-08-12,38,2,Ethan Clark,1,2026-02-22,15.82,...,counter,51782,3,1,4.25,3,Cold Brew 16oz,brew,4.25,0


# 1. Revenue and Units by Category

The two categories which reverse their relative order are "tea" and "pastry". We can see this by viewing our focussed query in descending order, where the tea had greater revenue but pastries actually had more units sold. This explains the reversal and can be traced back to the unit_price specified in order_items, as tea brings in more revenue since its unit price is higher than that of pastry.

In [ ]:
%%sql
SELECT
    p.category,
    SUM(oi.unit_price * oi.quantity) AS revenue,
    SUM(oi.quantity) AS units_sold
FROM products p
JOIN order_items oi ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC

,category,revenue,units_sold
0,espresso,197025.25,41410
1,brew,67355.30,18175
2,tea,44795.50,12196
3,pastry,39935.30,13168
4,food,21793.25,2963
5,merch,3310.00,334


In [ ]:
%%sql
SELECT
    p.category,
    SUM(oi.unit_price * oi.quantity) AS revenue,
    SUM(oi.quantity) AS units_sold
FROM products p
JOIN order_items oi ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY units_sold DESC

,category,revenue,units_sold
0,espresso,197025.25,41410
1,brew,67355.30,18175
2,pastry,39935.30,13168
3,tea,44795.50,12196
4,food,21793.25,2963
5,merch,3310.00,334


# 2. App-order share by Cafe

The difference between the lowest and highest earning cafes shows that there is a near 0 (~1%) difference in app share even if app orders vary in number across all cafes. We can defend the statement that app order shares are largely similar across all cafes.

In [ ]:
%%sql
SELECT s.store_id, s.name, COUNT(o.channel) as total_orders,
SUM(CASE WHEN o.channel = 'app' THEN 1 ELSE 0 END) as app_orders,
ROUND((SUM(CASE WHEN o.channel = 'app' THEN 1 ELSE 0 END)), 3) / COUNT(o.channel) AS app_share
FROM stores s
JOIN orders o ON o.store_id = s.store_id
GROUP BY s.store_id
ORDER BY app_share

,store_id,name,total_orders,app_orders,app_share
0,5,State Street East,6059,1788,0.295098
1,3,Discovery Park,4822,1423,0.295106
2,2,PMU Ground Floor,16916,5072,0.299834
3,4,Levee Plaza,7003,2110,0.301299
4,6,Airport Rd Drive-Thru,8258,2498,0.302495
5,1,Chauncey Hill,8869,2702,0.304657


In [ ]:
%%sql
WITH ranked_cafes AS(
SELECT
  s.store_id, s.name as cafe_name,
  ROUND((SUM(CASE WHEN o.channel = 'app' THEN 1 ELSE 0 END)), 6) / COUNT(o.channel) AS app_share,
  ROW_NUMBER() OVER (ORDER BY ROUND(SUM(CASE WHEN o.channel = 'app' THEN 1 ELSE 0 END) * 1.0 / COUNT(o.channel), 6) DESC, s.name) AS rank
  FROM stores s
  JOIN orders o ON o.store_id = s.store_id
  GROUP BY s.store_id
)

--inspect CTE
--SELECT * FROM ranked_cafes

SELECT
  MAX(CASE WHEN rank = 1 THEN cafe_name END) as highest_cafe,
  MAX(CASE WHEN rank = 1 THEN app_share END) as highest_share,
  MIN(CASE WHEN rank = 6 THEN cafe_name END) as loewst_cafe,
  MIN(CASE WHEN rank = 6 THEN app_share END) as lowest_share,
  ROUND(MAX(app_share) - MIN(app_share), 6) as difference
FROM ranked_cafes


,highest_cafe,highest_share,loewst_cafe,lowest_share,difference
0,Chauncey Hill,0.304657,State Street East,0.295098,0.009558


# 3. Highest-spending Customers

In [ ]:
%%sql
SELECT c.customer_id, c.name, o.order_id,
CAST(SUM(oi.unit_price*oi.quantity) AS int) as Spending
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
WHERE c.customer_id IS NOT NULL
GROUP BY c.customer_id, c.name
ORDER BY Spending DESC
LIMIT 3

,customer_id,name,order_id,Spending
0,439,Wei Shah,672,628
1,360,Priya Nguyen,858,601
2,259,Mateo Kim,325,594


# 4. Average Spending per Order by Loyalty Tier

Average spending per order by loyalty tier. First calculate each order’s value. For each of gold, silver, none, and walk-in, report average order value and number of orders; NULL customer_id means walk-in.

What do the averages show about spending per order?
Why can this result alone not establish that the program causes more visits, greater total spending
per customer, or more profit?

In [ ]:
%%sql
WITH order_totals AS (
    -- Calculate the total value of every order
    SELECT
        order_id,
        SUM(unit_price * quantity) AS total_value
    FROM order_items
    GROUP BY order_id
),
tier_data AS (
    -- Join totals with order/customer info to assign tiers
    SELECT
        ot.order_id,
        CASE
            WHEN o.customer_id IS NULL THEN 'walk-in'
            WHEN c.loyalty_tier IS NULL THEN 'none'
            ELSE c.loyalty_tier
        END AS tier,
        ot.total_value
    FROM orders o
    JOIN order_totals ot ON o.order_id = ot.order_id
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    -- Note: We want to keep nulls from the orders, so customers must be LEFT JOIN
)

SELECT
    tier,
    AVG(total_value) AS avg_order_value,
    COUNT(order_id) AS num_orders
FROM tier_data
GROUP BY tier
ORDER BY avg_order_value DESC;

,tier,avg_order_value,num_orders
0,silver,7.236427,10698
1,gold,7.205865,6496
2,none,7.202574,16530
3,walk-in,7.192850,18203


# 5. Above-average Revenue Months

Use a common table expression (WITH ... AS(...), often called a CTE) to calculate revenue for each of the twelve YYYY-MM months. Report each month whose revenue exceeds the average of those twelve totals, with its revenue. For this question, use fall = September–December, spring = January–April, and summer = May–August.

Which of these periods contain the returned months?



In [ ]:
%%sql
WITH months AS(
  SELECT
    order_ts, order_id,
    CASE
      WHEN strftime('%m', order_ts) IN ('09') THEN 'Sept'
      WHEN strftime('%m', order_ts) IN ('10') THEN 'Oct'
      WHEN strftime('%m', order_ts) IN ('11') THEN 'Nov'
      WHEN strftime('%m', order_ts) IN ('12') THEN 'Dec'
      WHEN strftime('%m', order_ts) IN ('01') THEN 'Jan'
      WHEN strftime('%m', order_ts) IN ('02') THEN 'Feb'
      WHEN strftime('%m', order_ts) IN ('03') THEN 'Mar'
      WHEN strftime('%m', order_ts) IN ('04') THEN 'Apr'
      WHEN strftime('%m', order_ts) IN ('05') THEN 'May'
      WHEN strftime('%m', order_ts) IN ('06') THEN 'Jun'
      WHEN strftime('%m', order_ts) IN ('07') THEN 'Jul'
      WHEN strftime('%m', order_ts) IN ('08') THEN 'Aug'
    END AS month
    FROM orders o
),
seasons AS(
  SELECT order_ts, order_id,
    CASE
      WHEN strftime('%m', order_ts) IN ('09', '10', '11', '12') THEN 'Fall'
      WHEN strftime('%m', order_ts) IN ('01', '02', '03', '04') THEN 'Spring'
      ELSE 'Summer'
    END AS season
  FROM orders o
),
-- define revenue as quantity * unit price
revenue AS(
  SELECT SUM(oi.quantity * oi.unit_price) as revenue, order_id
  FROM order_items oi
  GROUP BY oi.order_id
),
-- define monthly revenue by grouping by month
monthly_revenue AS(
  SELECT m.month, s.season, SUM(r.revenue) as monthly_revenue
  FROM months m
  JOIN revenue r ON r.order_id = m.order_id
  JOIN seasons s ON s.order_id = m.order_id
  GROUP BY m.month
)
-- final query
SELECT * FROM monthly_revenue
WHERE monthly_revenue > (SELECT AVG(monthly_revenue) FROM monthly_revenue)
ORDER BY season ASC

,month,season,monthly_revenue
0,Dec,Fall,31841.50
1,Nov,Fall,38093.40
2,Oct,Fall,40557.75
3,Sept,Fall,39629.65
4,Apr,Spring,36347.95
5,Feb,Spring,33479.20
6,Mar,Spring,36437.90


In [ ]:
%%sql
SELECT
    SUM(quantity * unit_price) AS total_dataset_revenue,
    SUM(quantity * unit_price) / 12.0 AS total_avg_monthly_revenue
FROM order_items
-- CHECK

,total_dataset_revenue,total_avg_monthly_revenue
0,374214.6,31184.55
